In [2]:
pip install --upgrade tensorflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install tensorflow

   ---------------------------------------- 0.0/331.9 MB ? eta -:--:--
    --------------------------------------- 4.2/331.9 MB 19.4 MB/s eta 0:00:17
   - -------------------------------------- 9.4/331.9 MB 22.6 MB/s eta 0:00:15
   - -------------------------------------- 14.2/331.9 MB 22.8 MB/s eta 0:00:14
   -- ------------------------------------- 18.6/331.9 MB 22.6 MB/s eta 0:00:14
   -- ------------------------------------- 24.1/331.9 MB 23.5 MB/s eta 0:00:14
   --- ------------------------------------ 28.8/331.9 MB 23.4 MB/s eta 0:00:13
   --- ------------------------------------ 32.8/331.9 MB 23.1 MB/s eta 0:00:13
   ---- ----------------------------------- 38.0/331.9 MB 23.2 MB/s eta 0:00:13
   ----- ---------------------------------- 43.0/331.9 MB 23.4 MB/s eta 0:00:13
   ----- ---------------------------------- 48.8/331.9 MB 23.9 MB/s eta 0:00:12
   ------ --------------------------------- 54.5/331.9 MB 24.1 MB/s eta 0:00:12
   ------- -------------------------------- 60.3/33

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
pip install numpy matplotlib opencv-python pillow

   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 8.1/8.1 MB 50.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 2.3/2.3 MB 32.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import collections
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
# from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

class ReliabilityValidator:
    def __init__(self, model_path='my_resnet_model.keras', window_size=10, alpha=0.4, min_conf=0.60):
        # 모델 로드 (ResNet 추천)
        self.model = load_model(
            model_path,
            custom_objects={'preprocess_input': resnet_preprocess}
        )
        self.window_size = window_size
        self.alpha = alpha          # EWMA 계수
        self.min_conf = min_conf    # 최소 신뢰도 필터
        
        # 큐: (class_idx, confidence, probabilities)
        self.queue = collections.deque(maxlen=window_size)
        
        # EWMA용 이전 값
        self.ewma_score = 0.0
        self.status = 0  # 0:Green, 1:Orange, 2:Red

    def preprocess(self, img_array):
        """(224,224,3) → 모델 입력 전처리"""
        img_array = np.expand_dims(img_array, axis=0)
        return resnet_preprocess(img_array)   # ResNet 기준 (MobileNet 필요시 변경)

    def predict_single(self, img_array):
        """단일 프레임 예측"""
        processed = self.preprocess(img_array)
        probs = self.model.predict(processed, verbose=0)[0]
        class_idx = np.argmax(probs)
        confidence = probs[class_idx]
        return class_idx, confidence, probs

    def update(self, img_array):
        """프레임 하나 추가 → 상태 갱신"""
        class_idx, conf, probs = self.predict_single(img_array)
        
        # 최소 신뢰도 필터
        if conf < self.min_conf:
            class_idx = 1  # Negative 강제 (패널티)
            conf = 0.5
        
        self.queue.append((class_idx, conf, probs))
        
        # 투표 계산
        if len(self.queue) == 0:
            return self.status
        
        positive_count = sum(1 for c, _, _ in self.queue if c in [0, 2])  # Fire or Smoke
        positive_ratio = positive_count / len(self.queue)
        
        # EWMA 스코어 (positive_ratio 평활화)
        self.ewma_score = self.alpha * positive_ratio + (1 - self.alpha) * self.ewma_score
        
        # 상태 결정
        if self.ewma_score >= 0.80:
            self.status = 2  # Red
        elif self.ewma_score >= 0.55:
            self.status = 1  # Orange
        else:
            self.status = 0  # Green
        
        return self.status, self.ewma_score, positive_ratio

    def get_status_text(self):
        return ["🟢 정상 (Green)", "🟠 경계 (Orange)", "🔴 위험 (Red)"][self.status]

# ==================== 사용 예시 ====================
if __name__ == "__main__":
    validator = ReliabilityValidator(model_path='my_resnet_model.keras', window_size=10)
    
    # 예: 10프레임 연속 처리 (실제로는 CCTV 프레임 루프)
    for i in range(15):   # 테스트용
        # img_array = ... (실제 프레임 로드)
        # status, ewma, ratio = validator.update(img_array)
        # print(f"Frame {i+1:2d} | Status: {validator.get_status_text()} | EWMA: {ewma:.3f} | Positive: {ratio:.1%}")
        pass

## 랜덤 노이즈 이미지로 테스트

Q: 실제 이미지 인가? A: 아님

dummy_input = np.random.rand(1, 224, 224, 3) * 255.0 는 더미 이미지를 만드는 코드

In [8]:
import os
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input

# 경고 억제 (혹시 남아있을 경우 대비)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

# 모델 경로 (본인 파일명으로 변경)
MODEL_PATH = 'my_resnet_model.keras'   # 또는 'my_mobilenet_model.keras'

print("모델 로드 중...")
model = load_model(
    MODEL_PATH,
    custom_objects={'preprocess_input': preprocess_input}
)
print("모델 로드 완료!")

# 간단한 더미 입력으로 예측 테스트
dummy_input = np.random.rand(1, 224, 224, 3) * 255.0  # 임의 이미지 모양
dummy_input = preprocess_input(dummy_input)

probs = model.predict(dummy_input, verbose=0)[0]
class_idx = np.argmax(probs)
confidence = probs[class_idx] * 100

class_names = ['Fire', 'Negative', 'Smoke']
print(f"더미 입력 예측 결과: {class_names[class_idx]} ({confidence:.2f}%)")

모델 로드 중...
모델 로드 완료!
더미 입력 예측 결과: Negative (84.23%)


모델 로드 완료한 결과

더미 입력 예측 결과: Negative (84.23%)

대부분의 분류 모델은 “이해할 수 없는 입력”이 들어오면 배경/정상 클래스로 분류하는 경향이 강하지만, 이 모델은 완전한 랜덤 이미지에 대해 Negative(정상)로 판단

## 실제 이미지로 테스트

ReliabilityValidator

In [ ]:
import os
import tensorflow as tf
import numpy as np
from collections import deque
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing import image
from pathlib import Path

# ==================== 경고 완전 억제 ====================
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')


class ReliabilityValidator:
    """E-gle Eye - Reliability Validation Layer
    최근 7프레임 슬라이딩 윈도우 + EWMA + 최소 신뢰도 필터링
    """

    def __init__(self, model_path='my_resnet_model.keras', window_size=7, alpha=0.4, min_conf=0.60):
        self.model = load_model(model_path, compile=False,
                                custom_objects={'preprocess_input': preprocess_input})
        self.window_size = window_size
        self.alpha = alpha
        self.min_conf = min_conf
        self.queue = deque(maxlen=window_size)   # bool: Positive(Fire/Smoke)
        self.ewma_score = 0.0
        self.status = 0  # 0:Green, 1:Orange, 2:Red

    def _is_positive(self, img_array):
        """단일 프레임 예측 + 최소 신뢰도 필터링"""
        processed = preprocess_input(np.expand_dims(img_array, axis=0))
        probs = self.model.predict(processed, verbose=0)[0]
        class_idx = np.argmax(probs)
        conf = probs[class_idx]
        return conf >= self.min_conf and class_idx in {0, 2}  # 0:Fire, 2:Smoke

    def update(self, img_array):
        """프레임 업데이트 → 상태 격상"""
        is_positive = self._is_positive(img_array)
        self.queue.append(is_positive)

        positive_ratio = sum(self.queue) / len(self.queue) if self.queue else 0.0
        self.ewma_score = self.alpha * positive_ratio + (1 - self.alpha) * self.ewma_score

        # Status Mapping
        self.status = 2 if self.ewma_score >= 0.80 else 1 if self.ewma_score >= 0.55 else 0
        return self.status, self.ewma_score, positive_ratio

    def get_status_text(self):
        return ["🟢 정상 (Green)", "🟠 경계 (Orange)", "🔴 위험 (Red)"][self.status]


# ==================== 테스트 실행 ====================
if __name__ == "__main__":
    validator = ReliabilityValidator()

    BASE = Path(r'E:\새 폴더\Egle_project\pytuning')

    # 동적 테스트 이미지 수집 (정상 → 연기 → 화재 순서 유지, 코드 대폭 단축)
    test_images = []
    for cat in ['normal', 'smoke', 'fire']:
        test_images.extend(sorted((BASE / 'testfiles' / cat).glob('*.jpg'))[:4])

    print("=== E-gle Eye 시계열 신뢰도 테스트 (Window=7) ===\n")

    for i, path in enumerate(test_images, 1):
        try:
            img_array = image.img_to_array(image.load_img(str(path), target_size=(224, 224)))
            status, ewma, ratio = validator.update(img_array)

            print(f"Frame {i:2d} | {path.name:30s} → {validator.get_status_text()} "
                  f"| EWMA {ewma:.3f} | Positive {ratio:.1%}")
        except Exception as e:
            print(f"Frame {i:2d} | {path.name} → 오류: {e}")

=== E-gle Eye 시계열 신뢰도 테스트 (Window=7) ===

Frame  1 | normal_image1.jpg              → 🟢 정상 (Green) | EWMA 0.000 | Positive 0.0%
Frame  2 | normal_image2.jpg              → 🟢 정상 (Green) | EWMA 0.000 | Positive 0.0%
Frame  3 | normal_image3.jpg              → 🟢 정상 (Green) | EWMA 0.000 | Positive 0.0%
Frame  4 | normal_image4.jpg              → 🟢 정상 (Green) | EWMA 0.000 | Positive 0.0%
Frame  5 | smoke_train1.jpg               → 🟢 정상 (Green) | EWMA 0.080 | Positive 20.0%
Frame  6 | smoke_train2.jpg               → 🟢 정상 (Green) | EWMA 0.181 | Positive 33.3%
Frame  7 | smoke_train3.jpg               → 🟢 정상 (Green) | EWMA 0.280 | Positive 42.9%
Frame  8 | smoke_train4.jpg               → 🟢 정상 (Green) | EWMA 0.397 | Positive 57.1%
Frame  9 | fire_image1.jpg                → 🟢 정상 (Green) | EWMA 0.524 | Positive 71.4%
Frame 10 | fire_image2.jpg                → 🟠 경계 (Orange) | EWMA 0.600 | Positive 71.4%
Frame 11 | fire_image3.jpg                → 🟠 경계 (Orange) | EWMA 0.646 | Positive 71.4%
Fra

In [3]:
import os
import tensorflow as tf
import numpy as np
from collections import deque
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing import image
from pathlib import Path

# ==================== 경고 완전 억제 ====================
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')


class ReliabilityValidator:
    """E-gle Eye - Reliability Validation Layer v2.0"""

    def __init__(self, model_path='my_resnet_model.keras', window_size=7, alpha=0.55, min_conf=0.60):
        self.model = load_model(model_path, compile=False,
                                custom_objects={'preprocess_input': preprocess_input})
        self.window_size = window_size
        self.alpha = alpha
        self.min_conf = min_conf
        self.queue = deque(maxlen=window_size)      # bool Positive
        self.ewma_score = 0.0
        self.status = 0

    def _predict_info(self, img_array):
        """상세 예측 정보 반환"""
        processed = preprocess_input(np.expand_dims(img_array, axis=0))
        probs = self.model.predict(processed, verbose=0)[0]
        class_idx = np.argmax(probs)
        conf = probs[class_idx]
        labels = ["화재", "정상", "연기"]
        is_positive = (conf >= self.min_conf) and class_idx in {0, 2}
        return labels[class_idx], round(conf, 3), is_positive

    def update(self, img_array):
        label, conf, is_positive = self._predict_info(img_array)
        self.queue.append(is_positive)

        positive_ratio = sum(self.queue) / len(self.queue) if self.queue else 0.0
        self.ewma_score = self.alpha * positive_ratio + (1 - self.alpha) * self.ewma_score

        # Hybrid Voting Logic (스펙 + 반응성 강화)
        if positive_ratio >= 0.80 or self.ewma_score >= 0.80:
            self.status = 2
        elif positive_ratio >= 0.55 or self.ewma_score >= 0.55:
            self.status = 1
        else:
            self.status = 0

        return self.status, self.ewma_score, positive_ratio, label, conf

    def get_status_text(self):
        return ["🟢 정상 (Green)", "🟠 경계 (Orange)", "🔴 위험 (Red)"][self.status]


# ==================== 테스트 실행 ====================
if __name__ == "__main__":
    validator = ReliabilityValidator()

    BASE = Path(r'E:\새 폴더\Egle_project\pytuning')
    test_images = []
    for cat in ['normal', 'smoke', 'fire']:
        test_images.extend(sorted((BASE / 'testfiles' / cat).glob('*.jpg'))[:4])

    print("=== E-gle Eye 시계열 신뢰도 테스트 v2.0 (Window=7) ===\n")

    for i, path in enumerate(test_images, 1):
        try:
            img_array = image.img_to_array(image.load_img(str(path), target_size=(224, 224)))
            status, ewma, ratio, label, conf = validator.update(img_array)

            print(f"Frame {i:2d} | {path.name:25s} → {validator.get_status_text()} "
                  f"| EWMA {ewma:.3f} | Ratio {ratio:.1%} | 예측: {label} ({conf:.2f})")
        except Exception as e:
            print(f"Frame {i:2d} | {path.name} → 오류: {e}")

=== E-gle Eye 시계열 신뢰도 테스트 v2.0 (Window=7) ===

Frame  1 | normal_image1.jpg         → 🟢 정상 (Green) | EWMA 0.000 | Ratio 0.0% | 예측: 정상 (1.00)
Frame  2 | normal_image2.jpg         → 🟢 정상 (Green) | EWMA 0.000 | Ratio 0.0% | 예측: 정상 (0.89)
Frame  3 | normal_image3.jpg         → 🟢 정상 (Green) | EWMA 0.000 | Ratio 0.0% | 예측: 정상 (1.00)
Frame  4 | normal_image4.jpg         → 🟢 정상 (Green) | EWMA 0.000 | Ratio 0.0% | 예측: 정상 (1.00)
Frame  5 | smoke_train1.jpg          → 🟢 정상 (Green) | EWMA 0.110 | Ratio 20.0% | 예측: 연기 (1.00)
Frame  6 | smoke_train2.jpg          → 🟢 정상 (Green) | EWMA 0.233 | Ratio 33.3% | 예측: 연기 (1.00)
Frame  7 | smoke_train3.jpg          → 🟢 정상 (Green) | EWMA 0.340 | Ratio 42.9% | 예측: 연기 (1.00)
Frame  8 | smoke_train4.jpg          → 🟠 경계 (Orange) | EWMA 0.468 | Ratio 57.1% | 예측: 연기 (0.97)
Frame  9 | fire_image1.jpg           → 🟠 경계 (Orange) | EWMA 0.603 | Ratio 71.4% | 예측: 연기 (0.98)
Frame 10 | fire_image2.jpg           → 🟠 경계 (Orange) | EWMA 0.664 | Ratio 71.4% | 예측: 정상 (1.00)
Fram

In [ ]:
import os
import tensorflow as tf
import numpy as np
from collections import deque
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing import image
from pathlib import Path

# ==================== 경고 완전 억제 ====================
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')


class ReliabilityValidator:
    """E-gle Eye - Reliability Validation Layer v4.0
    Fire Boost + Streak + 70% Red Threshold (골든타임 최적화)
    """

    def __init__(self, model_path='my_resnet_model.keras', window_size=7, alpha=0.58, min_conf=0.60):
        self.model = load_model(model_path, compile=False,
                                custom_objects={'preprocess_input': preprocess_input})
        self.window_size = window_size
        self.alpha = alpha
        self.min_conf = min_conf
        self.queue = deque(maxlen=window_size)
        self.ewma_score = 0.0
        self.streak = 0
        self.status = 0

    def _predict_info(self, img_array):
        """상세 예측 + Fire 강제 Positive"""
        processed = preprocess_input(np.expand_dims(img_array, axis=0))
        probs = self.model.predict(processed, verbose=0)[0]
        class_idx = np.argmax(probs)
        conf = probs[class_idx]
        labels = ["화재", "정상", "연기"]

        is_positive = (conf >= self.min_conf and class_idx in {0, 2})
        if class_idx == 0:                     # 화재 예측 시 강제 Positive
            is_positive = True

        return labels[class_idx], round(conf, 3), is_positive, class_idx

    def update(self, img_array):
        label, conf, is_positive, class_idx = self._predict_info(img_array)
        self.queue.append(is_positive)

        if is_positive:
            self.streak += 1
        else:
            self.streak = 0

        positive_ratio = sum(self.queue) / len(self.queue) if self.queue else 0.0
        self.ewma_score = self.alpha * positive_ratio + (1 - self.alpha) * self.ewma_score

        if (positive_ratio >= 0.70 or self.ewma_score >= 0.75 or self.streak >= 4):
            self.status = 2
        elif (positive_ratio >= 0.55 or self.ewma_score >= 0.55 or self.streak >= 2):
            self.status = 1
        else:
            self.status = 0

        return self.status, self.ewma_score, positive_ratio, self.streak, label, conf

    def get_status_text(self):
        return ["🟢 정상 (Green)", "🟠 경계 (Orange)", "🔴 위험 (Red)"][self.status]




In [4]:
import os
import tensorflow as tf
import numpy as np
import collections
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing import image
from pathlib import Path

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

class ReliabilityValidator:
    def __init__(self, model_path='my_resnet_model.keras', window_size=10, alpha=0.65, min_conf=0.55):
        self.model = load_model(model_path, custom_objects={'preprocess_input': preprocess_input})
        self.window_size = window_size
        self.alpha = alpha
        self.min_conf = min_conf
        self.queue = collections.deque(maxlen=window_size)
        self.ewma_score = 0.0
        self.status = 0

    def preprocess(self, img_array):
        img_array = np.expand_dims(img_array, axis=0)
        return preprocess_input(img_array)

    def predict_single(self, img_array):
        processed = self.preprocess(img_array)
        probs = self.model.predict(processed, verbose=0)[0]
        class_idx = np.argmax(probs)
        confidence = probs[class_idx]
        return class_idx, confidence, probs   # probs 추가 반환

    def update(self, img_array):
        class_idx, conf, probs = self.predict_single(img_array)
        if conf < self.min_conf:
            class_idx = 1
        self.queue.append((class_idx, conf))

        positive_count = sum(1 for c, _ in self.queue if c in [0, 2])
        positive_ratio = positive_count / len(self.queue)
        self.ewma_score = self.alpha * positive_ratio + (1 - self.alpha) * self.ewma_score

        if self.ewma_score >= 0.70:      # Red
            self.status = 2
        elif self.ewma_score >= 0.48:    # Orange
            self.status = 1
        else:
            self.status = 0
        return self.status, self.ewma_score, positive_ratio, class_idx

    def get_status_text(self):
        return ["🟢 정상 (Green)", "🟠 경계 (Orange)", "🔴 위험 (Red)"][self.status]

# ==================== 테스트 실행 ====================
if __name__ == "__main__":
    validator = ReliabilityValidator(model_path='my_resnet_model.keras', window_size=10)
    
    BASE = Path(r'E:\새 폴더\Egle_project\pytuning')
    
    test_images = [
        BASE / 'testfiles' / 'normal' / 'normal_image1.jpg',
        BASE / 'testfiles' / 'normal' / 'normal_image2.jpg',
        BASE / 'testfiles' / 'normal' / 'normal_image3.jpg',
        BASE / 'testfiles' / 'normal' / 'normal_image4.jpg',
        BASE / 'testfiles' / 'smoke' / 'smoke_train1.jpg',
        BASE / 'testfiles' / 'smoke' / 'smoke_train2.jpg',
        BASE / 'testfiles' / 'smoke' / 'smoke_train3.jpg',
        BASE / 'testfiles' / 'smoke' / 'smoke_train4.jpg',
        BASE / 'testfiles' / 'fire' / 'fire_image1.jpg',
        BASE / 'testfiles' / 'fire' / 'fire_image2.jpg',
        BASE / 'testfiles' / 'fire' / 'fire_image3.jpg',
        BASE / 'testfiles' / 'fire' / 'fire_image4.jpg',
    ]
    
    print("=== E-gle Eye v2.4 튜닝 테스트 (Window=10) ===\n")
    
    for i, path in enumerate(test_images):
        if not path.exists():
            print(f"Frame {i+1:2d} | {path.name} → 파일 없음")
            continue
        img = image.load_img(str(path), target_size=(224, 224))
        img_array = image.img_to_array(img)
        status, ewma, ratio, pred_class = validator.update(img_array)
        class_name = ['Fire', 'Negative', 'Smoke'][pred_class]
        
        print(f"Frame {i+1:2d} | {path.name:25s} | Pred: {class_name:8s} → {validator.get_status_text()} "
              f"| EWMA {ewma:.3f} | Positive {ratio:.1%}")

=== E-gle Eye v2.4 튜닝 테스트 (Window=10) ===

Frame  1 | normal_image1.jpg         | Pred: Negative → 🟢 정상 (Green) | EWMA 0.000 | Positive 0.0%
Frame  2 | normal_image2.jpg         | Pred: Negative → 🟢 정상 (Green) | EWMA 0.000 | Positive 0.0%
Frame  3 | normal_image3.jpg         | Pred: Negative → 🟢 정상 (Green) | EWMA 0.000 | Positive 0.0%
Frame  4 | normal_image4.jpg         | Pred: Negative → 🟢 정상 (Green) | EWMA 0.000 | Positive 0.0%
Frame  5 | smoke_train1.jpg          | Pred: Smoke    → 🟢 정상 (Green) | EWMA 0.130 | Positive 20.0%
Frame  6 | smoke_train2.jpg          | Pred: Smoke    → 🟢 정상 (Green) | EWMA 0.262 | Positive 33.3%
Frame  7 | smoke_train3.jpg          | Pred: Smoke    → 🟢 정상 (Green) | EWMA 0.370 | Positive 42.9%
Frame  8 | smoke_train4.jpg          | Pred: Smoke    → 🟢 정상 (Green) | EWMA 0.455 | Positive 50.0%
Frame  9 | fire_image1.jpg           | Pred: Smoke    → 🟠 경계 (Orange) | EWMA 0.520 | Positive 55.6%
Frame 10 | fire_image2.jpg           | Pred: Negative → 🟠 경계 (Orange)

In [14]:
import sys
import os
import tensorflow as tf
import numpy as np
from collections import deque
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing import image

# ==================== 경고 완전 억제 ====================
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')


# ==================== 프로젝트 루트 정확 설정 ====================
project_root = r'E:\newfolder\Egle_project\pytuning'   # ← 사용자님 현재 실제 경로
sys.path.append(project_root)

print("=== E-gle Eye 시계열 신뢰도 테스트 v9.0 (Hysteresis + 자동 경로 정리) ===\n")
print(f"📍 project_root : {project_root}")
print(f"   존재 여부     : {os.path.exists(project_root)}\n")


class ReliabilityValidator:
    """E-gle Eye - Reliability Validation Layer v9.0"""

    def __init__(self, model_path='my_resnet_model.keras', window_size=7, alpha=0.58, min_conf=0.60):
        self.model = load_model(model_path, compile=False,
                                custom_objects={'preprocess_input': preprocess_input})
        self.window_size = window_size
        self.alpha = alpha
        self.min_conf = min_conf
        self.queue = deque(maxlen=window_size)
        self.ewma_score = 0.0
        self.streak = 0
        self.status = 0

    def _predict_info(self, img_array):
        processed = preprocess_input(np.expand_dims(img_array, axis=0))
        probs = self.model.predict(processed, verbose=0)[0]
        class_idx = np.argmax(probs)
        conf = probs[class_idx]
        labels = ["화재", "정상", "연기"]

        is_positive = (conf >= self.min_conf and class_idx in {0, 2})
        if class_idx == 0:                     # 화재 예측 시 강제 Positive
            is_positive = True

        return labels[class_idx], round(conf, 3), is_positive

    def update(self, img_array):
        label, conf, is_positive = self._predict_info(img_array)
        self.queue.append(is_positive)

        if is_positive:
            self.streak += 1
        else:
            self.streak = 0

        positive_ratio = sum(self.queue) / len(self.queue) if self.queue else 0.0
        self.ewma_score = self.alpha * positive_ratio + (1 - self.alpha) * self.ewma_score

        previous_status = self.status

        # v9.0 Threshold (조금 더 안정적)
        if (positive_ratio >= 0.75 or self.ewma_score >= 0.78 or self.streak >= 4):
            self.status = 2
        elif (positive_ratio >= 0.55 or self.ewma_score >= 0.55 or self.streak >= 2):
            self.status = 1
        else:
            self.status = 0

        # ==================== Red Hysteresis (안정성 핵심) ====================
        if previous_status == 2 and positive_ratio >= 0.50:
            self.status = 2   # Red 상태 유지 (관리자 즉각 신고 유도)

        return self.status, self.ewma_score, positive_ratio, self.streak, label, conf

    def get_status_text(self):
        return ["🟢 정상 (Green)", "🟠 경계 (Orange)", "🔴 위험 (Red)"][self.status]


# ==================== 테스트 실행 ====================
if __name__ == "__main__":
    validator = ReliabilityValidator()

    BASE = project_root
    # 자동 정리: BASE가 testfiles로 잘못 붙었을 경우 상위 폴더로 수정
    if os.path.basename(BASE) == 'testfiles':
        BASE = os.path.dirname(BASE)
        print("⚠️ BASE가 testfiles로 설정되어 자동으로 상위 폴더로 수정했습니다.\n")

    test_dir = os.path.join(BASE, 'testfiles')

    print(f"✅ 최종 BASE 경로 : {BASE}")
    print(f"   존재 여부      : {os.path.exists(BASE)}")
    print(f"📁 testfiles 경로 : {test_dir}")
    print(f"   존재 여부      : {os.path.exists(test_dir)}\n")

    # 이미지 수집
    image_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.JPG', '.JPEG', '.PNG', '.BMP'}
    test_images = []
    search_dir = test_dir if os.path.exists(test_dir) else BASE
    for root, _, files in os.walk(search_dir):
        for file in files:
            if os.path.splitext(file)[1] in image_ext:
                test_images.append(os.path.join(root, file))

    test_images.sort()
    print(f"✅ 총 {len(test_images)}개 이미지 발견\n")

    # ==================== 발견된 이미지 미리 출력 ====================
    print("📋 발견된 테스트 이미지 목록:")
    for p in test_images:
        print(f"   - {os.path.basename(p)}")
    print("\n")

    if not test_images:
        print("❌ 이미지를 찾을 수 없습니다.")
    else:
        for i, full_path in enumerate(test_images, 1):
            filename = os.path.basename(full_path)
            try:
                img_array = image.img_to_array(image.load_img(full_path, target_size=(224, 224)))
                status, ewma, ratio, streak, label, conf = validator.update(img_array)

                print(f"Frame {i:2d} | {filename:28s} → {validator.get_status_text()} "
                      f"| EWMA {ewma:.3f} | Ratio {ratio:.1%} | Streak {streak} | 예측: {label} ({conf:.2f})")
            except Exception as e:
                print(f"Frame {i:2d} | {filename} → 로드 오류: {e}")

=== E-gle Eye 시계열 신뢰도 테스트 v9.0 (Hysteresis + 자동 경로 정리) ===

📍 project_root : E:\newfolder\Egle_project\pytuning
   존재 여부     : True

✅ 최종 BASE 경로 : E:\newfolder\Egle_project\pytuning
   존재 여부      : True
📁 testfiles 경로 : E:\newfolder\Egle_project\pytuning\testfiles
   존재 여부      : True

✅ 총 16개 이미지 발견

📋 발견된 테스트 이미지 목록:
   - test1.jpg
   - test10.jpg
   - test11.jpg
   - test12.jpg
   - test13.jpg
   - test14.jpg
   - test15.jpg
   - test16.jpg
   - test2.jpg
   - test3.jpg
   - test4.jpg
   - test5.png
   - test6.jpg
   - test7.jpg
   - test8.jpg
   - test9.jpg


Frame  1 | test1.jpg                    → 🔴 위험 (Red) | EWMA 0.580 | Ratio 100.0% | Streak 1 | 예측: 연기 (0.62)
Frame  2 | test10.jpg                   → 🔴 위험 (Red) | EWMA 0.824 | Ratio 100.0% | Streak 2 | 예측: 연기 (0.98)
Frame  3 | test11.jpg                   → 🔴 위험 (Red) | EWMA 0.926 | Ratio 100.0% | Streak 3 | 예측: 연기 (1.00)
Frame  4 | test12.jpg                   → 🔴 위험 (Red) | EWMA 0.824 | Ratio 75.0% | Streak 0 | 예측: 정상 (0.47

In [ ]:
import os
import tensorflow as tf
import numpy as np
import collections
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing import image
from pathlib import Path

# ==================== 경고 완전 억제 ====================
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

print(f"현재 작업 디렉토리: {os.getcwd()}\n")

# ==================== ReliabilityValidator 클래스 ====================
class ReliabilityValidator:
    def __init__(self, model_path='my_resnet_model.keras', window_size=10, alpha=0.4, min_conf=0.60):
        self.model = load_model(model_path, custom_objects={'preprocess_input': preprocess_input})
        self.window_size = window_size
        self.alpha = alpha
        self.min_conf = min_conf
        self.queue = collections.deque(maxlen=window_size)
        self.ewma_score = 0.0
        self.status = 0

    def preprocess(self, img_array):
        img_array = np.expand_dims(img_array, axis=0)
        return preprocess_input(img_array)

    def predict_single(self, img_array):
        processed = self.preprocess(img_array)
        probs = self.model.predict(processed, verbose=0)[0]
        class_idx = np.argmax(probs)
        confidence = probs[class_idx]
        return class_idx, confidence

    def update(self, img_array):
        class_idx, conf = self.predict_single(img_array)
        if conf < self.min_conf:
            class_idx = 1
        self.queue.append((class_idx, conf))

        positive_count = sum(1 for c, _ in self.queue if c in [0, 2])
        positive_ratio = positive_count / len(self.queue)
        self.ewma_score = self.alpha * positive_ratio + (1 - self.alpha) * self.ewma_score

        if self.ewma_score >= 0.80:
            self.status = 2
        elif self.ewma_score >= 0.55:
            self.status = 1
        else:
            self.status = 0
        return self.status, self.ewma_score, positive_ratio

    def get_status_text(self):
        return ["🟢 정상 (Green)", "🟠 경계 (Orange)", "🔴 위험 (Red)"][self.status]

# ==================== 테스트 실행 ====================
if __name__ == "__main__":
    validator = ReliabilityValidator(model_path='my_resnet_model.keras', window_size=10)
    
    # ★★★ (본인의 정확한 절대경로 그대로) ★★★
    BASE = Path(r'E:\새 폴더\Egle_project\pytuning')
    
    test_images = [
        # 정상 장면 4장
        BASE / 'testfiles' / 'normal' / 'normal_image1.jpg',
        BASE / 'testfiles' / 'normal' / 'normal_image2.jpg',
        BASE / 'testfiles' / 'normal' / 'normal_image3.jpg',
        BASE / 'testfiles' / 'normal' / 'normal_image4.jpg',

        # 연기 장면 4장
        BASE / 'testfiles' / 'smoke' / 'smoke_train1.jpg',
        BASE / 'testfiles' / 'smoke' / 'smoke_train2.jpg',
        BASE / 'testfiles' / 'smoke' / 'smoke_train3.jpg',
        BASE / 'testfiles' / 'smoke' / 'smoke_train4.jpg',

        # 화재 장면 4장
        BASE / 'testfiles' / 'fire' / 'fire_image1.jpg',
        BASE / 'testfiles' / 'fire' / 'fire_image2.jpg',
        BASE / 'testfiles' / 'fire' / 'fire_image3.jpg',
        BASE / 'testfiles' / 'fire' / 'fire_image4.jpg',
    ]
    
    print("=== E-gle Eye 시계열 신뢰도 테스트 (Window=10) ===\n")
    
    for i, path in enumerate(test_images):
        if not path.exists():
            print(f"Frame {i+1:2d} | {path.name:30s} → ❌ 파일 없음")
            continue
            
        try:
            img = image.load_img(str(path), target_size=(224, 224))
            img_array = image.img_to_array(img)
            status, ewma, ratio = validator.update(img_array)
            
            print(f"Frame {i+1:2d} | {path.name:30s} → {validator.get_status_text()} "
                  f"| EWMA {ewma:.3f} | Positive {ratio:.1%}")
        except Exception as e:
            print(f"Frame {i+1:2d} | {path.name} → 오류: {e}")

현재 작업 디렉토리: e:\새 폴더\Egle_project\pytuning

=== E-gle Eye 시계열 신뢰도 테스트 (Window=10) ===

Frame  1 | normal_image1.jpg              → 🟢 정상 (Green) | EWMA 0.000 | Positive 0.0%
Frame  2 | normal_image2.jpg              → 🟢 정상 (Green) | EWMA 0.000 | Positive 0.0%
Frame  3 | normal_image3.jpg              → 🟢 정상 (Green) | EWMA 0.000 | Positive 0.0%
Frame  4 | normal_image4.jpg              → 🟢 정상 (Green) | EWMA 0.000 | Positive 0.0%
Frame  5 | smoke_train1.jpg               → 🟢 정상 (Green) | EWMA 0.080 | Positive 20.0%
Frame  6 | smoke_train2.jpg               → 🟢 정상 (Green) | EWMA 0.181 | Positive 33.3%
Frame  7 | smoke_train3.jpg               → 🟢 정상 (Green) | EWMA 0.280 | Positive 42.9%
Frame  8 | smoke_train4.jpg               → 🟢 정상 (Green) | EWMA 0.368 | Positive 50.0%
Frame  9 | fire_image1.jpg                → 🟢 정상 (Green) | EWMA 0.443 | Positive 55.6%
Frame 10 | fire_image2.jpg                → 🟢 정상 (Green) | EWMA 0.466 | Positive 50.0%
Frame 11 | fire_image3.jpg                → 🟢 정상

window_size 분석 및 정리
프레임 단위의 이미지 분석을 위한 코드 작성 및 정상 작동 테스트
파일 경로를 정상적으로 불러오는지 테스트
실제 화재 영상에서 이미지를 추출하여 프레임단위로 올바르게 추출되는지 테스트
프레임에 맞춰 fire/smoke/normal 상태를 구별할 수 있는지 테스트 진행중


In [10]:
import sys 
import os
# dataset 폴더가 있는 상위 디렉토리를 sys.path에 추가 
project_root = r"E:\newfolder\Egle_project\pytuning\testfiles" 
sys.path.append(project_root)
# 확인용 (실행 후 출력되는지 보세요) 
print("현재 sys.path:") 
for p in sys.path: 
    print(p)

현재 sys.path:
e:\new\python312.zip
e:\new\DLLs
e:\new\Lib
e:\new

C:\Users\원준\AppData\Roaming\Python\Python312\site-packages
e:\new\Lib\site-packages
E:\newfolder\Egle_project\pytuning\testfiles


In [12]:
# ==================== 테스트 실행 v8.0 ====================
if __name__ == "__main__":
    validator = ReliabilityValidator()

    BASE = project_root
    test_dir = os.path.join(BASE, 'testfiles')

    print(f"\n✅ 최종 BASE 경로 : {BASE}")
    print(f"   존재 여부      : {os.path.exists(BASE)}")
    print(f"📁 testfiles 경로 : {test_dir}")
    print(f"   존재 여부      : {os.path.exists(test_dir)}\n")

    # 이미지 수집
    image_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.JPG', '.JPEG', '.PNG', '.BMP'}
    test_images = []

    search_dir = test_dir if os.path.exists(test_dir) else BASE
    for root, _, files in os.walk(search_dir):
        for file in files:
            if os.path.splitext(file)[1] in image_ext:
                test_images.append(os.path.join(root, file))

    test_images.sort()
    print(f"✅ 총 {len(test_images)}개 이미지 발견\n")

    if not test_images:
        print("❌ 이미지를 찾을 수 없습니다.")
        print("   → Explorer에서 testfiles 폴더 주소창 전체 복사 후 다시 실행해주세요!")
    else:
        for i, full_path in enumerate(test_images, 1):
            filename = os.path.basename(full_path)
            try:
                img_array = image.img_to_array(image.load_img(full_path, target_size=(224, 224)))
                status, ewma, ratio, streak, label, conf = validator.update(img_array)

                print(f"Frame {i:2d} | {filename:28s} → {validator.get_status_text()} "
                      f"| EWMA {ewma:.3f} | Ratio {ratio:.1%} | Streak {streak} | 예측: {label} ({conf:.2f})")
            except Exception as e:
                print(f"Frame {i:2d} | {filename} → 로드 오류: {e}")



✅ 최종 BASE 경로 : E:\newfolder\Egle_project\pytuning\testfiles
   존재 여부      : True
📁 testfiles 경로 : E:\newfolder\Egle_project\pytuning\testfiles\testfiles
   존재 여부      : False

✅ 총 16개 이미지 발견

Frame  1 | test1.jpg                    → 🔴 위험 (Red) | EWMA 0.580 | Ratio 100.0% | Streak 1 | 예측: 연기 (0.62)
Frame  2 | test10.jpg                   → 🔴 위험 (Red) | EWMA 0.824 | Ratio 100.0% | Streak 2 | 예측: 연기 (0.98)
Frame  3 | test11.jpg                   → 🔴 위험 (Red) | EWMA 0.926 | Ratio 100.0% | Streak 3 | 예측: 연기 (1.00)
Frame  4 | test12.jpg                   → 🔴 위험 (Red) | EWMA 0.824 | Ratio 75.0% | Streak 0 | 예측: 정상 (0.47)
Frame  5 | test13.jpg                   → 🟠 경계 (Orange) | EWMA 0.694 | Ratio 60.0% | Streak 0 | 예측: 정상 (1.00)
Frame  6 | test14.jpg                   → 🟠 경계 (Orange) | EWMA 0.678 | Ratio 66.7% | Streak 1 | 예측: 연기 (0.88)
Frame  7 | test15.jpg                   → 🟠 경계 (Orange) | EWMA 0.616 | Ratio 57.1% | Streak 0 | 예측: 연기 (0.49)
Frame  8 | test16.jpg                   → 🟢 정상

# 영상 이미지 테스트

In [16]:
import os
import cv2
import time
from pathlib import Path
import collections
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

class ReliabilityValidator:
    def __init__(self, model_path='my_resnet_model.keras', window_size=10, alpha=0.75):
        self.model = load_model(model_path, custom_objects={'preprocess_input': preprocess_input})
        self.window_size = window_size
        self.alpha = alpha
        self.queue = collections.deque(maxlen=window_size)
        self.ewma_score = 0.0
        self.status = 0

    def update(self, frame):
        # 프레임 전처리
        img = cv2.resize(frame, (224, 224))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_array = np.expand_dims(img, axis=0)
        processed = preprocess_input(img_array)

        probs = self.model.predict(processed, verbose=0)[0]
        class_idx = np.argmax(probs)
        conf = probs[class_idx]

        if conf < 0.55:
            class_idx = 1
            
        self.queue.append((class_idx, conf))

        positive_count = sum(1 for c, _ in self.queue if c in [0, 2])
        positive_ratio = positive_count / len(self.queue)
        self.ewma_score = self.alpha * positive_ratio + (1 - self.alpha) * self.ewma_score

        if self.ewma_score >= 0.65:
            self.status = 2  # Red
        elif self.ewma_score >= 0.45:
            self.status = 1  # Orange
        else:
            self.status = 0  # Green

        return self.status, self.ewma_score, positive_ratio, class_idx, conf*100

    def get_status_text(self):
        return ["🟢 정상", "🟠 경계", "🔴 위험"][self.status]

# ==================== 실행 ====================
if __name__ == "__main__":
    validator = ReliabilityValidator(model_path='my_resnet_model.keras')
    
    # ←←← 여기만 본인 mp4 경로로 바꾸세요
    video_path = r'평택 5번 시내버스 화재.mp4'
    
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    
    print("=== E-gle Eye 실시간 영상 테스트 시작 ===\n")
    print("ESC 키를 누르면 종료됩니다.\n")
    
    frame_count = 0
    start_time = time.time()
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        frame_count += 1
        
        # 1초에 약 5프레임 정도만 검사 (너무 빠르면 로그가 많아짐)
        if frame_count % max(1, fps//5) == 0:
            status, ewma, ratio, pred_class, conf = validator.update(frame)
            class_name = ['Fire', 'Negative', 'Smoke'][pred_class]
            
            elapsed = time.time() - start_time
            print(f"Time {elapsed:5.1f}s | Frame {frame_count:4d} | Pred: {class_name:8s} ({conf:.1f}%) "
                  f"→ {validator.get_status_text()} | EWMA {ewma:.3f} | Positive {ratio:.1%}")
        
        # 영상 화면도 함께 보여줌
        cv2.imshow('E-gle Eye - 화재 테스트', frame)
        
        if cv2.waitKey(1) == 27:   # ESC 키
            break
    
    cap.release()
    cv2.destroyAllWindows()
    print("\n테스트 종료")

=== E-gle Eye 실시간 영상 테스트 시작 ===

ESC 키를 누르면 종료됩니다.


테스트 종료


In [17]:
import cv2
from pathlib import Path

# ==================== 디버깅 테스트 ====================
video_path = r'E:\newfolder\Egle_project\평택 5번 시내버스 화재.mp4'   # ← 본인 파일 경로

print("현재 작업 디렉토리:", Path.cwd())
print("영상 파일 존재 여부:", Path(video_path).exists())
print("영상 파일 절대경로:", Path(video_path).resolve())

cap = cv2.VideoCapture(video_path)

print("VideoCapture 열림 여부:", cap.isOpened())

if cap.isOpened():
    print("영상 정보:")
    print("  - 너비:", int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)))
    print("  - 높이:", int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    print("  - FPS :", cap.get(cv2.CAP_PROP_FPS))
else:
    print("❌ VideoCapture 열기 실패! (한글 경로 또는 파일 문제)")

cap.release()

현재 작업 디렉토리: E:\newfolder\Egle_project\pytuning
영상 파일 존재 여부: True
영상 파일 절대경로: E:\newfolder\Egle_project\평택 5번 시내버스 화재.mp4
VideoCapture 열림 여부: True
영상 정보:
  - 너비: 1280
  - 높이: 720
  - FPS : 30.0


In [25]:
import os
import cv2
import time
from pathlib import Path
import collections
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

class ReliabilityValidator:
    def __init__(self, model_path='my_resnet_model.keras', window_size=10, alpha=0.62):
        self.model = load_model(model_path, custom_objects={'preprocess_input': preprocess_input})
        self.window_size = window_size
        self.alpha = alpha
        self.queue = collections.deque(maxlen=window_size)
        self.ewma_score = 0.0
        self.status = 0

    def update(self, frame):
        img = cv2.resize(frame, (224, 224))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_array = np.expand_dims(img, axis=0)
        processed = preprocess_input(img_array)

        probs = self.model.predict(processed, verbose=0)[0]
        class_idx = np.argmax(probs)
        conf = probs[class_idx]

        if conf < 0.58:
            class_idx = 1

        # Fire에 1.5배 가중치 부여 (오탐 저감 핵심)
        weight = 1.5 if class_idx == 0 else 1.0
        self.queue.append((class_idx, conf, weight))

        # 가중 투표 계산
        positive_score = sum(weight for c, _, w in self.queue if c in [0, 2])
        total_weight = sum(w for _, _, w in self.queue)
        positive_ratio = positive_score / total_weight if total_weight > 0 else 0

        self.ewma_score = self.alpha * positive_ratio + (1 - self.alpha) * self.ewma_score

        if self.ewma_score >= 0.78:      # Red (더 엄격하게)
            self.status = 2
        elif self.ewma_score >= 0.52:    # Orange
            self.status = 1
        else:
            self.status = 0
        return self.status, self.ewma_score, positive_ratio, class_idx, conf*100

    def get_status_text(self):
        return ["🟢 정상", "🟠 경계", "🔴 위험"][self.status]

# ==================== 실행 ====================
if __name__ == "__main__":
    validator = ReliabilityValidator(model_path='my_resnet_model.keras')
    
    video_path = r'E:\newfolder\Egle_project\화재진압, 포말방사기의 위력.mp4'   # ← 본인 경로
    
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    print("=== E-gle Eye v3.0 오탐 저감 테스트 시작 ===\n")
    
    frame_count = 0
    start_time = time.time()
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        frame_count += 1
        
        if frame_count % max(1, int(fps//5)) == 0:
            status, ewma, ratio, pred, conf = validator.update(frame)
            class_name = ['Fire', 'Negative', 'Smoke'][pred]
            elapsed = time.time() - start_time
            
            print(f"Time {elapsed:6.1f}s | Frame {frame_count:5d} | "
                  f"Pred: {class_name:8s} ({conf:5.1f}%) → {validator.get_status_text()} | "
                  f"EWMA {ewma:.3f} | Positive {ratio:.1%}")
        
        if cv2.waitKey(1) == 27:
            break
    
    cap.release()
    print("\n테스트 종료")

=== E-gle Eye v3.0 오탐 저감 테스트 시작 ===

Time    1.6s | Frame     6 | Pred: Smoke    ( 90.4%) → 🟠 경계 | EWMA 0.620 | Positive 100.0%
Time    1.8s | Frame    12 | Pred: Smoke    ( 92.6%) → 🔴 위험 | EWMA 0.856 | Positive 100.0%
Time    2.1s | Frame    18 | Pred: Smoke    ( 89.7%) → 🔴 위험 | EWMA 0.945 | Positive 100.0%
Time    2.3s | Frame    24 | Pred: Smoke    ( 70.1%) → 🔴 위험 | EWMA 0.979 | Positive 100.0%
Time    2.5s | Frame    30 | Pred: Negative ( 53.6%) → 🔴 위험 | EWMA 0.868 | Positive 80.0%
Time    2.8s | Frame    36 | Pred: Negative ( 51.0%) → 🟠 경계 | EWMA 0.743 | Positive 66.7%
Time    3.1s | Frame    42 | Pred: Smoke    ( 77.6%) → 🟠 경계 | EWMA 0.725 | Positive 71.4%
Time    3.3s | Frame    48 | Pred: Smoke    ( 91.3%) → 🟠 경계 | EWMA 0.741 | Positive 75.0%
Time    3.6s | Frame    54 | Pred: Smoke    ( 77.2%) → 🟠 경계 | EWMA 0.764 | Positive 77.8%
Time    3.8s | Frame    60 | Pred: Smoke    ( 67.2%) → 🔴 위험 | EWMA 0.786 | Positive 80.0%
Time    4.1s | Frame    66 | Pred: Smoke    ( 96.0%) → 🔴 위험